# Gmail Triage Agent — Demo

This notebook reads the triage log produced by `scripts/run_triage.py` and visualises
the results. No API calls are made here — it only reads what the agent already wrote to disk.

**For a live demo:** run `python scripts/run_triage.py` in a terminal first, then execute this notebook.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, Markdown

REPO_ROOT = Path().resolve().parent
LOG_FILE  = REPO_ROOT / 'results' / 'gmail_triage' / 'triage_log.csv'

if not LOG_FILE.exists():
    print('No triage log found. Run the agent first:')
    print('  python scripts/run_triage.py')
else:
    df = pd.read_csv(LOG_FILE, parse_dates=['timestamp'])
    print(f'Loaded {len(df)} classified thread(s) from {LOG_FILE.name}')
    display(df.tail(10))

## Distribution of categories

In [ ]:
CATEGORY_COLOURS = {
    'Universidad': '#4C72B0',
    'Trabajo':     '#DD8452',
    'Personal':    '#55A868',
    'Newsletters': '#C44E52',
    'Spam':        '#8172B2',
    'Otro':        '#937860',
}

counts = df['category'].value_counts()
colours = [CATEGORY_COLOURS.get(c, '#999999') for c in counts.index]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(counts.index, counts.values, color=colours, edgecolor='white', linewidth=0.8)

for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
            str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('Gmail Triage — threads per category', fontsize=14, pad=12)
ax.set_ylabel('Number of threads')
ax.set_xlabel('Category')
ax.set_ylim(0, counts.max() * 1.2)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(REPO_ROOT / 'results' / 'gmail_triage' / 'triage_distribution.png', dpi=150)
plt.show()

## Latest classified threads (most recent run)

In [ ]:
last_run_time = df['timestamp'].max()
# Group threads classified within 10 minutes of the latest entry as 'last run'.
last_run_df = df[df['timestamp'] >= last_run_time - pd.Timedelta(minutes=10)].copy()
last_run_df = last_run_df.sort_values('timestamp', ascending=False)

display(Markdown(f'### {len(last_run_df)} thread(s) classified in the last run ({last_run_time.strftime("%Y-%m-%d %H:%M")})\n'))

for _, row in last_run_df.iterrows():
    colour = CATEGORY_COLOURS.get(row['category'], '#999')
    badge  = f'<span style="background:{colour};color:white;padding:2px 8px;border-radius:4px;font-size:0.85em">{row["category"]}</span>'
    display(Markdown(
        f"{badge} &nbsp; **{row['subject'][:70]}**  "
        f"<br><small>From: {row['sender']} &nbsp;|&nbsp; {row['n_msgs']} email(s)</small>"
    ))

## Agent log (last 20 lines)

In [ ]:
agent_log = REPO_ROOT / 'results' / 'gmail_triage' / 'agent.log'
if agent_log.exists():
    lines = agent_log.read_text(encoding='utf-8').splitlines()
    print('\n'.join(lines[-20:]))
else:
    print('No agent.log found yet.')